#### Audio Synthesis

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration, ParlerTTSConfig
from transformers import AutoTokenizer, set_seed

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_id = "parler-tts/parler-tts-mini-expresso"
tokenizer = AutoTokenizer.from_pretrained(model_id)
config = ParlerTTSConfig.from_pretrained(model_id)
config.decoder._attn_implementation = "flash_attention_2"
model = ParlerTTSForConditionalGeneration.from_pretrained(model_id, config=config).to(device)
set_seed(42)  # For reproducibility of results
print(f"Using device: {device}")

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import soundfile as sf

def synthesize_dataset(csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    try:
        df = pd.read_csv(csv_path, index_col=0)
    except FileNotFoundError:
        print(f"Error: Could not find {csv_path}. Please run the generation script first.")
        return

    audio_names = []
    for prompt_id, row in tqdm(df.iterrows(), total=len(df), desc="Generating Audio"):
        user_cmd = row["User_Command"]
        description = row["Voice_Description"]
        cmd_id = row["cmd_id"]
        # 1. Tokenize the acoustic description condition
        input = tokenizer(description, return_tensors="pt").to(device)
        # 2. Tokenize the semantic text to be spoken
        prompt = tokenizer(user_cmd, return_tensors="pt").to(device)
        # 3. Generate the acoustic waveform tensor
        # We explicitly set prompt_input_ids to map the text directly to the description
        generation = model.generate(input_ids=input.input_ids,
                                    prompt_input_ids=prompt.prompt_input_ids,
                                    do_sample=True, # Enable sampling for more natural variance
                                    temperature=1.1) # Increase temperature for more creative generation
        # Convert tensor to a numpy array
        audio_arr = generation.cpu().numpy().squeeze()
        # 4. Save to disk (Naming convention links it to the original command and variation)
        filename = f"prompt_{prompt_id}_cmd_{cmd_id}_varia_{prompt_id % 3}.wav"
        filepath = os.path.join(output_dir, f"prompt_{prompt_id}_cmd_{cmd_id}_varia_{prompt_id % 3}.wav")
        sf.write(filepath, audio_arr, model.config.sampling_rate)
        audio_names.append(filename)
    # 5. Save the updated dataframe with the audio paths
    df["audio_file_name"] = audio_names
    updated_csv_path = csv_path.replace(".csv", "_with_audio.csv")
    df.to_csv(updated_csv_path, index=False)

In [ ]:
synthesize_dataset("./data/cleaned_raw_train.csv", "./data/synthesized_train")
synthesize_dataset("./data/cleaned_raw_test.csv", "./data/synthesized_test")

In [ ]:
import gc

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

#### Acoustic Quality and Variance Assessment

Extracts F0, Energy, and Duration from generated .wav files to mathematically prove conditional acoustic variance.

In [ ]:
import librosa
import numpy as np

def evaluate_acoustic_variance(audio_dir, sample_limit=5000):
    audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
    # Limit sample size for faster evaluation if the dataset is massive
    if len(audio_files) > sample_limit:
        audio_files = np.random.choice(audio_files, sample_limit, replace=False)
        print(f"Randomly sampled {sample_limit} files for F0 extraction.")
    results = []
    for filename in tqdm(audio_files, desc="Extracting Acoustic Features"):
        filepath = os.path.join(audio_dir, filename)
        # Load audio
        y, sr = librosa.load(filepath, sr=None)
        # 1. Duration Analysis
        duration = librosa.get_duration(y=y, sr=sr)
        # 2. Energy Variance (Root Mean Square)
        rms = librosa.feature.rms(y=y)[0]
        mean_energy = np.mean(rms)
        # 3. Fundamental Frequency (F0) Extraction using pYIN algorithm
        # fmin=50Hz (deep male), fmax=500Hz (high female)
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        # Filter out unvoiced frames (silence/consonants) to get true pitch
        valid_f0 = f0[~np.isnan(f0)]
        mean_pitch = np.mean(valid_f0) if len(valid_f0) > 0 else 0
        pitch_std = np.std(valid_f0) if len(valid_f0) > 0 else 0
        results.append({
            "Filename": filename,
            "Duration (s)": duration,
            "Mean Energy (RMS)": mean_energy,
            "Mean Pitch (Hz)": mean_pitch,
            "Pitch Variance (Std)": pitch_std
        })
    df_results = pd.DataFrame(results)
    # 4. Statistical variance checks
    # To prove the model is not collapsing to a single voice, we want to see a wide range of values.
    print("\n--- Acoustic Variance Metrics ---")
    print("Goal: High standard deviation across the dataset indicates successful conditional synthesis.")
    pitch_dataset_std = df_results["Mean Pitch (Hz)"].std()
    energy_dataset_std = df_results["Mean Energy (RMS)"].std()
    dur_dataset_std = df_results["Duration (s)"].std()
    print(f"Dataset Pitch (F0) Std. Dev:   {pitch_dataset_std:.2f} Hz " + ("(PASS: High variance)" if pitch_dataset_std > 30 else "(FAIL: Mode Collapse)"))
    print(f"Dataset Energy (RMS) Std. Dev: {energy_dataset_std:.4f} " + ("(PASS: Volume variation detected)" if energy_dataset_std > 0.01 else "(WARNING: Monotone Volume)"))
    print(f"Dataset Duration Std. Dev:     {dur_dataset_std:.2f} sec")
    return df_results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_acoustic_variance(df_results):
    if df_results is None or df_results.empty:
        return

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Evaluation 1.2: Acoustic Quality and Variance Assessment', fontsize=16, fontweight='bold', y=1.05)

    # Plot 1: Pitch (F0) Distribution
    # A bimodal distribution here proves the model generated both distinct Male (low Hz) and Female (high Hz) voices
    sns.histplot(df_results["Mean Pitch (Hz)"], kde=True, ax=axes[0], color="purple", bins=30)
    axes[0].set_title('Fundamental Frequency (F0) Variance', fontweight='bold')
    axes[0].set_xlabel('Mean Pitch (Hz)')

    # Plot 2: Energy (RMS) Distribution
    sns.histplot(df_results["Mean Energy (RMS)"], kde=True, ax=axes[1], color="teal", bins=30)
    axes[1].set_title('Energy (RMS) Variance', fontweight='bold')
    axes[1].set_xlabel('Mean Energy / Amplitude')

    # Plot 3: Duration Distribution
    # Proves the "pacing" modifier (fast/slow) successfully altered the waveform length
    sns.histplot(df_results["Duration (s)"], kde=True, ax=axes[2], color="coral", bins=30)
    axes[2].set_title('Utterance Duration Variance', fontweight='bold')
    axes[2].set_xlabel('Duration (Seconds)')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_acoustic_variance(evaluate_acoustic_variance("./data/synthesized_train"))

In [ ]:
plot_acoustic_variance(evaluate_acoustic_variance("./data/synthesized_test"))